# A short introduction to containerized software

After spending time using nf-core pipelines to answer bioinformatic questions, we will focus on the processes that lie behind these pipelines now.

Today, we will focus on containerization, namely via Docker. 



1. Check if Docker is installed.

In [ ]:
!docker info

### What is a container?

A container is an isolated, running instance of an image that packages a tool with all its dependencies, so it runs the same on any machine.

### Why do we use containers?

We use containers to ensure the reproducibility of , e.g. pipelines. Each container has its own isolated environment, so tools with conflicting dependencies don't interfere with each other. Containers are also portable, so they run the same way on any computer.

### What is a docker image?

It is the blueprint to use for creating containers: a read-only package containing a tool, its dependencies and the operating system files that it needs.

### Let's run our first docker image:

### Login to docker

In [ ]:
# This you need to do on the command line directly

### Run your first docker container

In [ ]:
!docker run hello-world

### Find the container ID

ID: 1250c07ec7c1 

In [ ]:
!docker ps -a

### Delete the container again, give prove its deleted

In [ ]:
!docker rm -f 1250c07ec7c1
!docker inspect 1250c07ec7c1

*The output

(compWorkflow) pia@Pias-MacBook-Air day_04 % docker inspect 1250c07ec7c1
[]
error: no such object: 1250c07ec7c1

### FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line

Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/

Please describe the steps you took to download and run the software for the example fastq file from last week below:

1. Download the FastQC software (curl -L -O https://www.bioinformatics.babraham.ac.uk/projects/fastqc/fastqc_v0.12.1.zip)
2. Unzip and make executable (unzip fastqc_v0.12.1.zip
chmod +x FastQC/fastqc)
3.  Run FastQC on an example file (./FastQC/fastqc ../day_02/fetchngs_out/fastq/SRX19144486_SRR23195516_1.fastq.gz -o fastqc_results)
4. Analyse the quality metrics in the resulting HTML report

### Very well, now let's try to make use of its docker container

1. create a container holding fastqc using seqera containers (https://seqera.io/containers/)
2. use the container to generate a fastqc html of the example fastq file

In [ ]:
# pull the container
!docker pull community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae
!docker images

In [ ]:
# run the container and save the results to a new "fastqc_results" directory
!mkdir -p fastqc_results_seqera

!docker run --rm \
  -v "$(pwd)/..":/data \
  -w /data \
  community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae \
  fastqc day_02/fetchngs_out/fastq/SRX19144486_SRR23195516_1.fastq.gz \
  -o day_04/fastqc_results_seqera

### Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?

Running FastQC locally was easier this time, since there was nothing to learn beyond download and run. Docker will probably be easier in the future because it runs the same everywhere without installing anything, fixes the tool version for reproducibility, and can work directly with workflow managers like Nextflow.

### What would you say, which approach is more reproducible?

Docker, because the image pins the exact tool version and all dependencies, so it runs identically on any machine.

### Compare the file to last weeks fastqc results, are they identical?
### Is the fastqc version identical?

Yes they are identical. The underlying results (fastqc_data.txt) are identical. Only the HTML report differs slightly (e.g. creation date and embedded images), which doesn't affect the results.
Both runs used FastQC v0.12.1.


## Dockerfiles

We now used Docker containers and images directly to boost our research. 

Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay)

Hints:
1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [ ]:
# open the file "my_dockerfile" in a text editor
!code my_dockerfile

### Explain the RUN and ENV lines you added to the file

`RUN` executes commands while the image is being built. Here it updates the package list (`apt-get update`), installs cowsay (`apt-get install -y`, where `-y` confirms automatically because the build is not interactive) and deletes the package lists afterwards to keep the image small.
`ENV` sets an environment variable inside the image. cowsay is installed in `/usr/games`, which is not in the default PATH, so `ENV PATH="/usr/games:${PATH}"` adds it and `cowsay` can be called directly.

(Explanation for the salmon_docker file: RUN executes commands while the image is being built. The first RUN installs curl and ca-certificates (needed for HTTPS downloads). The second RUN downloads Salmon v1.5.2 with curl, unpacks it, moves the program to /usr/bin and its libraries to /usr/lib, and deletes the archive to keep the image small.
ENV sets an environment variable inside the image. Here it adds /usr/bin to PATH, so salmon can be called directly without its full path.)

In [ ]:
# build the docker image
!docker build -t my_cowsay -f my_dockerfile .

In [ ]:
# make sure that the image has been built
!docker images my_cowsay

#IMAGE              ID             DISK USAGE   CONTENT SIZE   EXTRA
#my_cowsay:latest   34f28c22ca93        194MB         43.5MB    

In [ ]:
# run the docker file 
!docker run --rm my_cowsay cowsay "Muuuuh"

## Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq

To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

In [ ]:
# use the file "salmon_docker" in this directory to build a new docker image

In [ ]:
# build the image
!docker build -t my_salmon -f salmon_docker .

In [ ]:
# run the docker image to give out the version of salmon
!docker run --rm my_salmon salmon --version

## Do you think bioinformaticians have to create a docker image every time they want to run a tool?

Find the salmon docker image online and run it on your computer.

What is https://biocontainers.pro/ ?

No, for most of the common tools images exist. You can just pull and run them from public websites like Seqera and Biocontainers.

BioContainers is a community project, that provides Docker containers for many bioinformatics tools. They are built automatically with Bioconda packages.

I found the Salmon image on BioContainers (quay.io/biocontainers/salmon) and pulled version 1.5.2 to run it locally.

In [ ]:
!docker pull quay.io/biocontainers/salmon:1.5.2--h84f40af_0
!docker run --rm quay.io/biocontainers/salmon:1.5.2--h84f40af_0 salmon --version

## Are there other ways to create Docker (or Apptainer) images?

What is https://seqera.io/containers/ ?

Yes. Instead of writing a Dockerfile, you can use Seqera Containers, a free web service where you select packages and it automatically builds a Docker or Apptainer image for you. The image is hosted publicly at community.wave.seqera.io and can be pulled directly, as we did for FastQC.